# FBRef web scraping — Big 5 wages

This notebook is the **only** script for scraping: it fetches **player-level salaries** and **team-level total wages** from FBRef for the Big 5 European leagues (2022–23, 2023–24, 2024–25). All logic lives here; there are no separate `.py` files to run.

**How to use:** Set the notebook working directory to `Code/Scraping/`, then **Run All**. The output of each cell will stay visible under the cell. Run in order: Setup first, then Section 1 (player wages), then Section 2 (team wages).

**Requirements:** `pandas`, `requests`. Optional: set env var `SCRAPER_API_KEY` for ScraperAPI (or use the default in the Setup cell). Outputs go to `Database/Player_Salaries/` and `Database/Club_Total_Wages/` at project root.

**Sections:**
1. **Setup** — Paths, API key, and shared helpers (`fetch`, `uncomment`, summary URL).
2. **Player wages by club** — For each season, loads the Big 5 summary page, follows each team link, parses the player wage table, and saves one CSV per season (e.g. `Big5_2022-2023_player_salaries.csv`).
3. **Team total wages** — For each season, parses the summary table (one row per team) and saves one CSV per season (e.g. `Big5_2022-2023_team_total_wages.csv`).

In [ ]:
import io
import os
import re
import time
from pathlib import Path
from typing import List, Tuple
from urllib.parse import quote

import pandas as pd
import requests

# Paths: notebook is in Code/Scraping/; project root = parent.
BASE = Path(".").resolve()  # Code/Scraping
PROJECT_ROOT = BASE.parent.parent
SALARIES_DIR = PROJECT_ROOT / "Database" / "Player_Salaries"
TEAM_WAGES_DIR = PROJECT_ROOT / "Database" / "Club_Total_Wages"

SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

# ScraperAPI key (optional). Set env SCRAPER_API_KEY or use default for testing.
SCRAPER_API_KEY = os.environ.get("SCRAPER_API_KEY", "2b0f166d2f91a3843c368d6c16245423")


def big5_wages_summary_url(season: str) -> str:
    """Big 5 wages summary page (table of teams with links to each squad's wage page)."""
    return f"https://fbref.com/en/comps/Big5/{season}/wages/{season}-Big-5-European-Leagues-Wages"


def fetch(url: str) -> str:
    """Fetch URL (via ScraperAPI if key is set)."""
    api_url = f"https://api.scraperapi.com?api_key={SCRAPER_API_KEY}&url={quote(url, safe='')}"
    r = requests.get(api_url, timeout=90)
    r.raise_for_status()
    return r.text


def uncomment(html: str) -> str:
    """Remove HTML comments so tables inside them are visible to pd.read_html."""
    return html.replace("<!--", "").replace("-->", "")

## 1. Player wages by club

For each season we load the Big 5 wages summary page, extract links to each team's wage page, then fetch each team page and parse the player wage table. Results are combined into one CSV per season in `Database/Player_Salaries/` (e.g. `Big5_2022-2023_player_salaries.csv`).

In [ ]:
def extract_squad_wage_links(html: str, season: str) -> List[Tuple[str, str, str]]:
    """Parse the Big 5 wages summary page for links to each team's wage page. Returns (squad_id, wage_slug, team_display_name)."""
    out = []
    wage_pattern = re.compile(
        r'href="/en/squads/([a-f0-9]+)/' + re.escape(season) + r'/wages/([^"/]+)"',
        re.I,
    )
    stats_pattern = re.compile(
        r'href="/en/squads/([a-f0-9]+)/' + re.escape(season) + r'/([^"/]+)-Stats"',
        re.I,
    )
    seen = set()
    for m in wage_pattern.finditer(html):
        sid, slug = m.group(1), m.group(2)
        key = (sid, slug)
        if key in seen:
            continue
        seen.add(key)
        team_name = slug.replace("-Wage-Details", "").replace("-Stats", "").strip().replace("-", " ")
        out.append((sid, slug, team_name))
    for m in stats_pattern.finditer(html):
        sid, base = m.group(1), m.group(2)
        wage_slug = base.replace("-Stats", "") + "-Wage-Details"
        key = (sid, wage_slug)
        if key in seen:
            continue
        seen.add(key)
        team_name = wage_slug.replace("-Wage-Details", "").replace("-", " ")
        out.append((sid, wage_slug, team_name))
    return out


def find_wages_table(html: str) -> pd.DataFrame | None:
    """Find the largest table that has Player and wage columns."""
    tables = pd.read_html(io.StringIO(html))
    best, best_len = None, 0
    for df in tables:
        if df.empty or len(df) < 3:
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df = df.copy()
            df.columns = [str(c[0]) if isinstance(c, tuple) and c[0] else str(c) for c in df.columns]
        cols = [str(c).lower() for c in df.columns]
        if not any("player" in c for c in cols) or not any("wage" in c or "annual" in c for c in cols):
            continue
        if len(df) > best_len:
            best, best_len = df, len(df)
    return best


SALARIES_DIR.mkdir(parents=True, exist_ok=True)

for season in SEASONS:
    print(f"Fetching {season}: Big 5 wages summary (team list)...")
    summary_url = big5_wages_summary_url(season)
    try:
        summary_html = fetch(summary_url)
    except Exception as e:
        print(f"  Error loading summary: {e}")
        continue
    squads = extract_squad_wage_links(summary_html, season)
    print(f"  Found {len(squads)} teams. Fetching each team's player wages...")
    time.sleep(1)
    all_dfs = []
    for i, (squad_id, wage_slug, team_name) in enumerate(squads):
        url = f"https://fbref.com/en/squads/{squad_id}/{season}/wages/{wage_slug}"
        try:
            club_html = fetch(url)
            club_html = uncomment(club_html)
            df = find_wages_table(club_html)
            if df is not None:
                first_col = df.columns[0]
                df = df[df[first_col].astype(str).str.lower() != "player"].copy()
                if not df.empty:
                    df = df.copy()
                    if "Team" in df.columns:
                        df["Team"] = team_name
                    else:
                        df.insert(0, "Team", team_name)
                    all_dfs.append(df)
        except Exception:
            pass
        if (i + 1) % 20 == 0:
            print(f"    {i + 1}/{len(squads)} teams done...")
        time.sleep(1)
    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)
        cols = [c for c in combined.columns if c != "Team"]
        combined = combined[["Team"] + cols]
        out_path = SALARIES_DIR / f"Big5_{season}_player_salaries.csv"
        combined.to_csv(out_path, index=False)
        print(f"  {season}: saved {len(combined)} players from {len(all_dfs)} teams -> {out_path.name}")
    else:
        print(f"  {season}: no data")

print("Done. Player salary CSVs in Database/Player_Salaries/")

## 2. Team total wages

For each season we fetch the same Big 5 wages summary page and parse the **summary table** (one row per team: Squad, Comp, Weekly Wages, Annual Wages, etc.). Results are saved to `Database/Club_Total_Wages/` (e.g. `Big5_2022-2023_team_total_wages.csv`).

In [ ]:
def find_team_wages_table(html: str) -> pd.DataFrame | None:
    """Find the table with squad totals (Squad, Comp, Weekly/Annual Wages)."""
    tables = pd.read_html(io.StringIO(html))
    for df in tables:
        if df.empty or len(df) < 5:
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df = df.copy()
            df.columns = [str(c[0]) if isinstance(c, tuple) and c[0] else str(c) for c in df.columns]
        cols = [str(c).lower() for c in df.columns]
        has_squad = any("squad" in c for c in cols)
        has_wage = any("wage" in c or "annual" in c for c in cols)
        if has_squad and has_wage:
            return df
    return None


TEAM_WAGES_DIR.mkdir(parents=True, exist_ok=True)

for season in SEASONS:
    print(f"Fetching {season} team wages...")
    url = big5_wages_summary_url(season)
    try:
        html = fetch(url)
        html = uncomment(html)
        df = find_team_wages_table(html)
        if df is not None:
            first_col = df.columns[0]
            s = df[first_col].astype(str).str.strip().str.lower()
            df = df[~(s.isin(["squad", "rk", "rank", ""]))].copy()
            out_path = TEAM_WAGES_DIR / f"Big5_{season}_team_total_wages.csv"
            df.to_csv(out_path, index=False)
            print(f"  Saved {len(df)} teams -> {out_path.name}")
        else:
            print(f"  No table found for {season}")
    except Exception as e:
        print(f"  Error: {e}")
    time.sleep(1)

print("Done. Team wage CSVs in Database/Club_Total_Wages/")